In [ ]:
import pandas as pd

In [ ]:
file_path = "/home/Harsh/Documents/Coding/Rohit/fake news classification/data/evaluation.csv"

df = pd.read_csv(file_path, sep=";")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
import re
import html
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


In [ ]:
for label, group in df.groupby("label"):
    print(f"\nLabel: {label}")
    display(group[["title", "text", "label"]].head(2))

In [ ]:
required_columns = {"title", "text", "label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.dropna(subset=["label"]).copy()

df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["content"] = (
    df["title"].str.strip() + " " + df["text"].str.strip()
).str.strip()

df = df[df["content"].ne("")].copy()

print("Shape after basic cleaning:", df.shape)

In [ ]:
df.head()

In [ ]:
df["text_length"] = df["content"].str.len()
df["word_count"] = df["content"].str.split().str.len()

display(df.groupby("label")[["text_length", "word_count"]].describe())

df.groupby("label")["word_count"].mean().plot(
    kind="bar",
    title="Average article word count by label",
    ylabel="Average word count"
)
plt.show()

In [ ]:
import re
import html
import spacy

nlp = spacy.load("en_core_web_sm")

NEGATIONS = {"no", "not", "nor", "never", "n't"}

def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def preprocess_texts(texts, batch_size=128):
    cleaned_texts = (clean_text(text) for text in texts)
    results = []

    for doc in nlp.pipe(cleaned_texts, batch_size=batch_size):
        tokens = []

        for token in doc:
            if token.is_space or token.is_punct:
                continue

            if token.is_stop and token.lower_ not in NEGATIONS:
                continue

            lemma = token.lemma_.lower().strip()

            if lemma:
                tokens.append(lemma)

        results.append(" ".join(tokens))

    return results

df["processed_text"] = preprocess_texts(df["content"].fillna(""))


In [ ]:
df["processed_text"]

In [ ]:
df.head()

In [ ]:
processed_df = df[["processed_text", "label"]].copy()

output_path = "/home/Harsh/Documents/Coding/Rohit/fake news classification/data/processed_fake_news.csv"

processed_df.to_csv(output_path, index=False, encoding="utf-8")

print("Processed dataset saved successfully!")

In [ ]:
fp = "/home/Harsh/Documents/Coding/Rohit/fake news classification/data/processed_fake_news.csv"
df = pd.read_csv(fp)

In [ ]:
import re, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
data = df[["processed_text", "label"]].copy()           
data = data.dropna()
data["processed_text"] = data["processed_text"].astype(str).str.strip()
data = data[data["processed_text"].str.len() > 20]

In [ ]:
import re

def clean(t):
    t = re.sub(r"^.{0,40}\((reuters|ap)\)\s*-?\s*", "", t, flags=re.I)
    t = re.sub(r"https?://\S+|@\w+", " ", t)
    return t
data["processed_text"] = data["processed_text"].map(clean)

In [ ]:
before = len(data)
data = data.drop_duplicates(subset="processed_text").reset_index(drop=True)
print(f"Removed {before - len(data)} duplicate/empty rows")
print("Label counts:\n", data["label"].value_counts()) 

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    data["processed_text"], data["label"], test_size=0.2,
    stratify=data["label"], random_state=42)
assert len(X_tr) == len(y_tr) and len(X_te) == len(y_te)

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3,
                              max_df=0.8, sublinear_tf=True, stop_words="english")),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced")),
])

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import loguniform, randint
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_pipe(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                                  stop_words="english")),
        ("clf", clf),
    ])

In [ ]:
tfidf_params = {
    "tfidf__max_features": [3000, 5000, 10000],
    "tfidf__min_df": [2, 3, 5],
    "tfidf__max_df": [0.7, 0.8, 0.9],
}

In [ ]:
models = {
    "Logistic Regression": {
        "pipe": make_pipe(LogisticRegression(max_iter=100, class_weight="balanced")),
        "params": {
            **tfidf_params,
            "clf__C": loguniform(1e-2, 1e2),
            "clf__penalty": ["l2"],
            "clf__solver": ["liblinear", "lbfgs"],
        },
    },
    "Random Forest": {
        "pipe": make_pipe(RandomForestClassifier(class_weight="balanced",
                                                 random_state=42, n_jobs=-1)),
        "params": {
            **tfidf_params,
            "tfidf__max_features": [3000, 5000],        
            "clf__n_estimators": randint(100, 200),
            "clf__max_depth": [10, 20, 40, None],
            "clf__min_samples_split": randint(2, 20),
            "clf__min_samples_leaf": randint(1, 10),     
            "clf__max_features": ["sqrt", "log2"],
        },
    },
}

results, best_models = [], {}

In [ ]:
for name, m in models.items():
    print(f"\n=== Tuning {name} ===")
    search = RandomizedSearchCV(
        m["pipe"], m["params"],
        n_iter=20,                 
        scoring="f1_macro",
        cv=cv, n_jobs=-1, random_state=42,
        return_train_score=True,   
        verbose=1,
    )
    search.fit(X_tr, y_tr)
    best_models[name] = search.best_estimator_

    idx = search.best_index_
    train_s = search.cv_results_["mean_train_score"][idx]
    cv_s = search.best_score_
    test_s = f1_score(y_te, search.predict(X_te), average="macro")

    results.append({
        "Model": name,
        "Train F1": round(train_s, 4),
        "CV F1": round(cv_s, 4),
        "Test F1": round(test_s, 4),
        "Overfit gap (train-CV)": round(train_s - cv_s, 4),
    })
    print("Best params:", search.best_params_)

summary = pd.DataFrame(results).set_index("Model")
print("\n=== Comparison ===")
print(summary)

In [ ]:
import numpy as np, pandas as pd

best = best_models["Logistic Regression"]
names = best["tfidf"].get_feature_names_out()
coef = best["clf"].coef_[0]
classes = best["clf"].classes_
print("Class order:", classes, "-> positive coef pushes toward", classes[1])

print("\nTop words toward", classes[1], ":", list(names[np.argsort(coef)[-20:]][::-1]))
print("Top words toward", classes[0], ":", list(names[np.argsort(coef)[:20]]))

# Look at actual mistakes
pred = best.predict(X_te)
errors = pd.DataFrame({"text": X_te.values, "true": y_te.values, "pred": pred})
errors = errors[errors.true != errors.pred]
print("\nTotal errors:", len(errors))
print(pd.crosstab(errors.true, errors.pred, rownames=["true"], colnames=["pred"]))
for t in errors.text.head(5):
    print("\n---", t[:300])

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import loguniform, randint
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

LEAK = {
    "reuters", "bloomberg", "getty", "images", "image", "feature", "featured",
    "wire", "century", "21st", "breitbart", "via", "source", "photo", "pic",
    "twitter", "facebook", "com", "www", "http", "https",
    "video", "watch", "read", "click", "share", "breaking", "subscribe",
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "sept",
    "oct", "nov", "dec",
    "don", "doesn", "didn", "isn", "wasn", "aren", "couldn", "wouldn",
    "shouldn", "won", "hasn", "haven", "hadn",
    "washington",
}
stops = list(ENGLISH_STOP_WORDS | LEAK)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_pipe(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                                  stop_words=stops)),
        ("clf", clf),
    ])

tfidf_params = {
    "tfidf__max_features": [3000, 5000, 10000],
    "tfidf__min_df": [3, 5, 10],
    "tfidf__max_df": [0.7, 0.8, 0.9],
}

models = {
    "Logistic Regression": {
        "pipe": make_pipe(LogisticRegression(max_iter=100, class_weight="balanced")),
        "params": {**tfidf_params,
                   "clf__C": loguniform(1e-2, 1e1),
                   "clf__solver": ["liblinear", "lbfgs"]},
    },
    "Random Forest": {
        "pipe": make_pipe(RandomForestClassifier(class_weight="balanced",
                                                 random_state=42, n_jobs=-1)),
        "params": {**tfidf_params,
                   "tfidf__max_features": [3000, 5000],
                   "clf__n_estimators": randint(100, 200),
                   "clf__max_depth": [10, 20, 40, None],
                   "clf__min_samples_split": randint(2, 20),
                   "clf__min_samples_leaf": randint(2, 10),
                   "clf__max_features": ["sqrt", "log2"]},
    },
}

results, best_models = [], {}
for name, m in models.items():
    print(f"\n=== Tuning {name} ===")
    search = RandomizedSearchCV(m["pipe"], m["params"], n_iter=20,
                                scoring="f1_macro", cv=cv, n_jobs=-1,
                                random_state=42, return_train_score=True, verbose=1)
    search.fit(X_tr, y_tr)
    best_models[name] = search.best_estimator_

    i = search.best_index_
    train_s = search.cv_results_["mean_train_score"][i]
    cv_s = search.best_score_
    test_s = f1_score(y_te, search.predict(X_te), average="macro")
    results.append({"Model": name, "Train F1": round(train_s, 4),
                    "CV F1": round(cv_s, 4), "Test F1": round(test_s, 4),
                    "Gap (train-CV)": round(train_s - cv_s, 4)})
    print("Best params:", search.best_params_)

summary = pd.DataFrame(results).set_index("Model")
print("\n=== Comparison (cleaned features) ===")
print(summary)

for name, est in best_models.items():
    print(f"\n--- {name} ---")
    p = est.predict(X_te)
    print(confusion_matrix(y_te, p))
    print(classification_report(y_te, p))

lr = best_models["Logistic Regression"]
ln = lr["tfidf"].get_feature_names_out()
c = lr["clf"].coef_[0]
print("\nLR top -> 1 (real):", list(ln[np.argsort(c)[-20:]][::-1]))
print("LR top -> 0 (fake):", list(ln[np.argsort(c)[:20]]))

rf = best_models["Random Forest"]
rn = rf["tfidf"].get_feature_names_out()
print("\nRF top features:", list(rn[np.argsort(rf["clf"].feature_importances_)[-20:]][::-1]))

In [ ]:
import pickle

final_model = best_models["Logistic Regression"]

tfidf_fitted = final_model["tfidf"]
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_fitted, f)

clf_fitted = final_model["clf"]
with open("lr_classifier.pkl", "wb") as f:
    pickle.dump(clf_fitted, f)

with open("fake_news_lr_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Saved: tfidf_vectorizer.pkl, lr_classifier.pkl, fake_news_lr_model.pkl")

with open("tfidf_vectorizer.pkl", "rb") as f:
    tfidf_loaded = pickle.load(f)
with open("lr_classifier.pkl", "rb") as f:
    clf_loaded = pickle.load(f)

X_te_vec = tfidf_loaded.transform(X_te)          #
pred_separate = clf_loaded.predict(X_te_vec)

assert (pred_separate == final_model.predict(X_te)).all()
print("Separate files match the pipeline. Vocabulary size:", len(tfidf_loaded.vocabulary_))

